# Graph Classification of Unseen Data

The goal is to classify one or more unseen graphs based on its **graph label**. The graph label is categorical and is one of [0,1,2,3,4]
The model is trained on the Building-Ground Relationshop dataset (BGR)

## What this notebook assumes

You should have already prepared dataset directory containing:

- `graphs.csv`
- `nodes.csv`
- `edges.csv`

### IMPORTANT: Make sure you manually assign graph labels for each graph in graphs.csv. The label is your assessment of the relationship of the building in that graph to the ground around it.
### The goal is to see if the pre-trained model agrees with you.
### Labels:

- 0: "Separation"
- 1: "Separation with Plinth"
- 2: "Adherence"
- 3: "Adherence with Plinth"
- 4: "Interlock"

## What this notebook does

1. Loads the prepared dataset
2. Loads a pre-trained model (given to you in notebooks/Support Files)
3. Predicts the test graphs
4. Visualizes the results

## 1. Imports and renderer

In [1]:
import pandas as pd
from pathlib import Path
from topologicpy.PyG import PyG

renderer = "vscode"

## 2. Set the dataset path, model path, and input parameters

In [4]:
DATASET_PATH = Path(r"C:\Users\Win11\GraphML_RaniaChihaoui\Tutorials\Supporting Files\dataset_graph_classification").resolve()
MODEL_PATH = Path(r"C:\Users\Win11\GraphML_RaniaChihaoui\Tutorials\Supporting Files\bgr_model.pt").resolve()
PREDICTION_LEVEL = "graph"
TASK = "classification"
GRAPH_LABEL_TYPE = "categorical"
NODE_LABEL_TYPE = "categorical"
EDGE_LABEL_TYPE = "categorical"
MAPPING = {0: "Separation",
           1: "Separation with Plinth",
           2: "Adherence",
           3: "Adherence with Plinth",
           4: "Interlock"}

## 3. Load the prepared dataset

In [5]:
pyg_2 = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType=GRAPH_LABEL_TYPE,
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType=EDGE_LABEL_TYPE,
    # If your headers differ, override here (your attached CSVs match defaults):
    # graphIDHeader="graph_id", graphLabelHeader="label",
    # nodeIDHeader="node_id", nodeLabelHeader="label",
    # edgeSRCHeader="src_id", edgeDSTHeader="dst_id", edgeLabelHeader="label",
)

## 4. Load the pre-trained model

In [6]:
pyg_2.LoadModel(str(MODEL_PATH))

## 5. Make the whole dataset a testing dataset

In [7]:
pyg_2.SetHyperparameters(split=(0.0, 0.0, 1.0), shuffle=False)  # all graphs become test
print("Done")

Done


## 6. Predict the dataset

In [8]:
pred_results = pyg_2.Predict()
predicted_values = pred_results['pred'].tolist()
predicted_labels = [MAPPING[pv] for pv in predicted_values]
actual_values = pred_results['y_true'].tolist()
actual_labels = [MAPPING[av] for av in actual_values]
probabilities = pred_results['prob'].tolist()

df = pd.DataFrame({
    "Actual Value": actual_values,
    "Predicted Value": predicted_values,
    "Actual Label": actual_labels,
    "Predicted Label": predicted_labels,
    "Confidence": [round(max(p), 2) for p in probabilities]
})

df

,Actual Value,Predicted Value,Actual Label,Predicted Label,Confidence
0,4,4,Interlock,Interlock,1.0
1,0,0,Separation,Separation,1.0
2,4,4,Interlock,Interlock,1.0
3,4,4,Interlock,Interlock,1.0
4,4,4,Interlock,Interlock,1.0
...,...,...,...,...,...
1491,1,1,Separation with Plinth,Separation with Plinth,1.0
1492,0,0,Separation,Separation,1.0
1493,1,1,Separation with Plinth,Separation with Plinth,1.0
1494,4,4,Interlock,Interlock,1.0
